# GegenPress-PPCF: Vectorized Potential Pitch Control & Risk Surface Engine
**Author:** Rangga  
**Repository:** [gegenpress-ppcf-engine](https://github.com/Rangga11268/gegenpress-ppcf-engine)  
**Tactical Domain:** Spatio-Temporal Physics, 4D SIMD Vectorization, Liverpool FC Gegenpress Dynamics & High-Defensive-Line Exposure

---

### Core Theoretical & Mathematical Foundations

Based on Dr. William Spearman's Potential Pitch Control Field (PPCF) framework (MIT Sloan Sports Analytics Conference, 2018), this engine models the probability that an attacking or defending team controls any spatial location $(x, y)$ on the pitch.

#### 1. Kinematic Time-to-Intercept ($T_i$)
For any player $i$ running toward pitch coordinate $(x, y)$, arrival time accounts for reaction latency $\tau_i$ and acceleration-limited sprinting:
$$T_i(x, y) = \tau_i + t_{\text{run}}(d_i)$$
$$t_{\text{run}}(d) = \begin{cases} \sqrt{\frac{2d}{a_{\max}}}, & d \le d_{\text{accel}} \\ \frac{v_{\max}}{a_{\max}} + \frac{d - d_{\text{accel}}}{v_{\max}}, & d > d_{\text{accel}} \end{cases} \quad \text{where } d_{\text{accel}} = \frac{v_{\max}^2}{2 a_{\max}}$$

#### 2. Vectorized Soft-Min Team Race Integration
Rather than running multi-step Euler integration over $1.5 \times 10^7$ grid points, we reformulate the race into a competitive Poisson process using numerical log-sum-exp soft-min:
$$T_{\text{team}}(x, y) = -\frac{1}{\beta} \ln \sum_{i \in \text{team}} \exp\left(-\beta (T_i - T_{\min})\right) + T_{\min}$$
$$PPCF_{\text{att}}(x, y) = \frac{1}{1 + \exp\left(-\beta (T_{\text{def}}(x, y) - T_{\text{att}}(x, y))\right)}$$

#### 3. Dominance Contraction Ratio (DCR)
Measures the rate at which an elite Gegenpressing unit collapses opponent-controlled territory within 3.0 seconds of turnover:
$$DCR = \frac{\text{Area}_{\text{opp}}(t=0) - \text{Area}_{\text{opp}}(t=3.0)}{\text{Area}_{\text{opp}}(t=0)}$$
- **Elite Threshold:** $DCR \ge 65.0\%$ in $\le 3.5\text{s}$.


In [ ]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath(os.path.dirname(__file__) if '__file__' in locals() else '.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from app.config import DEFAULT_PHYSICS, DEFAULT_PITCH
from app.core.ppcf_engine import PPCFEngine
from app.core.tactical_metrics import (
    compute_counter_attack_risk_surface,
    compute_cover_shadow_occlusion,
    compute_spatial_dominance_collapse,
)
from app.data.metrica_loader import generate_synthetic_turnover_scenario
from app.viz.pitch_renderer import render_ppcf_frame

print("GegenPress-PPCF modules initialized.")


### 1. Initialize Engine & Generate 75-Frame Gegenpress Turnover
Loads 75 consecutive frames (3.0 seconds @ 25 FPS) of Liverpool FC trapping an opponent ball carrier at Anfield.


In [ ]:
engine = PPCFEngine(pitch_cfg=DEFAULT_PITCH, physics_cfg=DEFAULT_PHYSICS)
frames = generate_synthetic_turnover_scenario(n_frames=75, fps=25.0)

t0 = time.perf_counter()
res_0 = engine.compute_frame_ppcf(frames[0], attack_direction=1, apply_offside_mask=False)
res_end = engine.compute_frame_ppcf(frames[-1], attack_direction=1, apply_offside_mask=False)
bench_time = (time.perf_counter() - t0) * 1000.0 / 2.0

print(f"Loaded {len(frames)} frames. Single-frame calculation: {bench_time:.2f} ms ({1000/bench_time:.1f} FPS)")


### 2. Spatial Dominance Collapse (DCR) Analysis
Verify that opponent space collapses by $\ge 65.0\%$ across the 3.0-second pressing sequence.


In [ ]:
dcr = compute_spatial_dominance_collapse(res_0, res_end)

print(f"Initial Opponent Area (t=0.0s): {dcr.initial_opp_area_m2:.1f} m²")
print(f"Final Opponent Area   (t=3.0s): {dcr.final_opp_area_m2:.1f} m²")
print(f"Dominance Contraction Ratio (DCR): {dcr.contraction_ratio * 100:.1f}%")
print(f"Elite Gegenpress Gate (>= 65.0%): {'PASSED (ELITE)' if dcr.is_elite_gegenpress else 'FAILED'}")


### 3. Geometric Cover Shadow Passing Lane Occlusion
Calculates angular shadow cones from pressing defenders to forward passing outlets.


In [ ]:
f0 = frames[0]
ball_pos = f0.ball_position
defenders = [p for p in f0.players if p.team == "att" and "darwin" in p.player_id or "endo" in p.player_id]
opp_targets = [p for p in f0.players if p.team == "def" and p.player_id in ("opp_cf_target", "opp_w_right", "opp_cm_right")]

shadows = compute_cover_shadow_occlusion(ball_pos, defenders, opp_targets)
for s in shadows:
    status_str = "BLOCKED" if s.is_occluded else "OPEN"
    print(f"Passing Lane to [{s.target_id}]: {status_str} (Angle: {s.shadow_angle_deg:.1f}°, Occlusion Prob: {s.occlusion_prob:.2f})")


### 4. Tactical Visualization of High-Line Pitch Control
Render the 105m x 68m spatial heatmap and player vectors using the pitch renderer.


In [ ]:
fig, ax = render_ppcf_frame(frames[-1], res_end, pitch_cfg=DEFAULT_PITCH)
plt.title("GegenPress PPCF Dominance Heatmap (t = 3.0s)", color='#F8FAFC', fontsize=12, pad=10)
plt.show()
